# grid_100x100 PIE パトロール（A* + SpotDog 往復）

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中** にこのノートを実行してください。

## シナリオ

1. 外周ブロックを実体化 (`SetBlocking True`)
2. Humanoid をマス **(5, 5)**、SpotDog を **(10, 5)** にスポーン
3. SpotDog が A* 経路で **(50, 50)** へ移動 → **5 秒停止**
4. 出発点 **(10, 5)** へ A* で帰還

ロジック: `grid_env_10k_pie_patrol.py`（コストマップ解像度 0.30 m = ブロック幅、`path_planning_costmap` の格子 A*）

カーネル: `conda activate simworld`

## 初回セットアップ（Editor PIE）

1. **Robot_Dog** を Editor プロジェクトへコピー（未導入時）:
   `bash dev/grid_env_10k/scripts/install_robot_dog_editor.sh`
2. UE Editor を再起動するか、**PIE を Stop → Play** してアセットを再読込
3. Outliner の `block_*` はラベル名。UnrealCV は `BP_TransparentCube_C_UAID_...` を使うため、本スクリプトが位置から自動解決します

## 同じ PIE セッションで再実行するとき

- `skip_perimeter_if_already_solid=True` … 外周 T 化をスキップ（約8分短縮）
- `skip_robot_probe=True` … BP プローブをスキップ（Rename クラッシュ回避）
- 前回の SpotDog が残っている場合は **再スポーンせずテレポート** で起点に戻します
- クラッシュした場合は PIE Stop → Play してから Cell 1 から再実行

In [1]:
import sys
print(sys.executable)

/home/winder17wsl_ishizawalab/miniforge3/envs/simworld/bin/python


In [3]:
import importlib
from pathlib import Path


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_g10k = _root / "dev" / "grid_env_10k"
for p in (_root, _g10k):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import grid_env_10k_pie_patrol as patrol

importlib.reload(patrol)
print(f"[Paths] root={_root}")

[Paths] root=/home/winder17wsl_ishizawalab/00_kotaprivate/Program/SimWorld


In [4]:
# マス番号 (gx, gy) — 1 始まり
HUMAN_CELL = (5, 5)
ROBOT_START = (10, 5)
ROBOT_GOAL = (50, 50)
GOAL_DWELL_S = 5.0

print(
    f"human={HUMAN_CELL}, robot {ROBOT_START} -> {ROBOT_GOAL}, dwell={GOAL_DWELL_S}s"
)

human=(5, 5), robot (10, 5) -> (50, 50), dwell=5.0s


In [5]:
# Re-run in the same PIE session: skip perimeter + probe; robot is teleported if still alive.
result = patrol.run_patrol_scenario(
    human_cell=HUMAN_CELL,
    robot_start_cell=ROBOT_START,
    robot_goal_cell=ROBOT_GOAL,
    goal_dwell_s=GOAL_DWELL_S,
    skip_perimeter_if_already_solid=True,
    skip_robot_probe=True,
)
result

[UE] Probing UnrealCV on ['127.0.0.1', '10.103.0.1', '10.255.255.254'] (timeout=3s each) ...
[UE] probe OK: 127.0.0.1:9000


INFO:__init__:238:Got connection confirm: b'connected to SimWorld'


[UE] probe skip: 10.103.0.1:9000
[UE] probe skip: 10.255.255.254:9000
=>Info: using ip-port socket
[UE] Connected via UnrealCV at 127.0.0.1:9000
[Pak] OK pakchunk1000-Windows.pak: Mounted C:\SimWorldServer\SimWorld\Content\Paks\pakchunk1000-Windows.pak
[Pak] OK pakchunk0-Windows.pak: Mounted C:\SimWorldServer\SimWorld\Content\Paks\pakchunk0-Windows.pak
[UE] spawn_bp_asset '__GridEnv_SpotRobot_probe__' (timeout=60s) ...
[UE] destroy existing '__GridEnv_SpotRobot_probe__' ...
[Registry] scanning 10000 cube actors ...
[Registry] cache saved: /home/winder17wsl_ishizawalab/00_kotaprivate/Program/SimWorld/dev/grid_env_10k/.pie_block_registry.json
[Registry] mapped 396/396 blocks in 0.0s
[Scenario] skip perimeter (assume already T)
[Costmap] 30 m, resolution=0.30 m, lethal from 396 blocks, grid=(100, 100)
[UE] destroy existing 'GridEnv_SpotRobot' ...
(135.0, 135.0, 100.0)
[Humanoid] GEN_BP_Humanoid_0 @ cell(5,5) map=(1.3499999999999999, 1.3499999999999999) world=(135.0, 135.0, 100.0)
[UE] spa

PatrolResult(perimeter_ok=True, human_name='GEN_BP_Humanoid_0', robot_spawned=False, outbound_arrived=False, return_arrived=False, start_xy=(285.0, 135.0), goal_xy=(1485.0, 1485.0), final_xy=(285.0, 135.0), return_dist_cm=inf)

In [ ]:
success = (
    result.perimeter_ok
    and result.robot_spawned
    and result.outbound_arrived
    and result.return_arrived
    and result.return_dist_cm <= patrol.RETURN_ARRIVE_TOLERANCE_CM
)
print(f"SUCCESS={success}, return_dist={result.return_dist_cm:.1f} cm")